# Harm-Aware Evaluation for Safety-Critical Systems

**Duration:** 30 minutes  
**Level:** Advanced  
**Prerequisites:** Complete [Notebook 03: Coverage-Risk](03_coverage_risk_selective_prediction.ipynb)

## Overview

**Harm-Aware Evaluation** asks: *Does the system minimize harm in high-risk scenarios?*

**Not all errors are equal.** Missing a high-risk patient (false negative) is typically more harmful than over-triaging a low-risk patient (false positive). Traditional accuracy treats all errors equally harm-aware metrics weight errors by their clinical consequences.

### What You'll Learn:

1. Weighted harm loss with custom harm weights
2. Per-tier harm analysis
3. Escalation failure detection
4. Harm concentration metrics
5. Asymmetric cost matrices

### Why This Matters:

**Clinical reality:**
- Missing a critical patient: potentially fatal
- Over-triaging a healthy patient: unnecessary resource use
- **Harm weights reflect clinical priorities**

## 1. Setup

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import BASICS-CDSS harm-aware module
from basics_cdss.metrics.harm import (
    weighted_harm_loss,
    harm_by_risk_tier,
    escalation_failure_analysis,
    harm_concentration_index,
    compute_harm_metrics,
    asymmetric_cost_matrix,
    DEFAULT_HARM_WEIGHTS
)
from basics_cdss.visualization.harm_plots import (
    plot_harm_by_tier,
    plot_escalation_analysis,
    plot_harm_concentration
)

print("✓ BASICS-CDSS harm-aware module loaded")
print(f"\nDefault harm weights: {DEFAULT_HARM_WEIGHTS}")

## 2. Generate Mock CDSS Data

We'll create predictions with different error patterns to demonstrate harm-aware metrics.

In [ ]:
np.random.seed(42)
n_samples = 500

# Generate risk tiers
risk_tiers = np.random.choice(
    ['high', 'medium', 'low'],
    size=n_samples,
    p=[0.20, 0.30, 0.50]
)

# True labels (1 = needs intervention, 0 = defer)
y_true = np.zeros(n_samples, dtype=int)
y_true[risk_tiers == 'high'] = np.random.binomial(1, 0.80, (risk_tiers == 'high').sum())
y_true[risk_tiers == 'medium'] = np.random.binomial(1, 0.40, (risk_tiers == 'medium').sum())
y_true[risk_tiers == 'low'] = np.random.binomial(1, 0.10, (risk_tiers == 'low').sum())

# Model A: Balanced errors across all tiers
y_pred_balanced = np.copy(y_true)
error_indices = np.random.choice(n_samples, size=int(n_samples * 0.15), replace=False)
y_pred_balanced[error_indices] = 1 - y_pred_balanced[error_indices]  # Flip labels

# Model B: Errors concentrated in high-risk tier (DANGEROUS)
y_pred_high_risk_errors = np.copy(y_true)
high_risk_idx = np.where(risk_tiers == 'high')[0]
high_risk_errors = np.random.choice(high_risk_idx, size=int(len(high_risk_idx) * 0.25), replace=False)
y_pred_high_risk_errors[high_risk_errors] = 1 - y_pred_high_risk_errors[high_risk_errors]

# Model C: Errors only in low-risk tier (safer)
y_pred_low_risk_errors = np.copy(y_true)
low_risk_idx = np.where(risk_tiers == 'low')[0]
low_risk_errors = np.random.choice(low_risk_idx, size=int(len(low_risk_idx) * 0.20), replace=False)
y_pred_low_risk_errors[low_risk_errors] = 1 - y_pred_low_risk_errors[low_risk_errors]

print(f"Generated {n_samples} CDSS predictions")
print(f"\nRisk tier distribution:")
print(pd.Series(risk_tiers).value_counts())

print(f"\nModel Accuracies:")
print(f"  Balanced:            {(y_pred_balanced == y_true).mean():.1%}")
print(f"  High-risk errors:    {(y_pred_high_risk_errors == y_true).mean():.1%}")
print(f"  Low-risk errors:     {(y_pred_low_risk_errors == y_true).mean():.1%}")

print("\nNote: Similar accuracies, but very different clinical impact!")

## 3. Weighted Harm Loss

Weighted harm loss assigns different weights to errors based on risk tier:

$$
L_{\text{harm}} = \frac{1}{N} \sum_{i=1}^{N} w_{r_i} \cdot \mathbb{1}[\hat{y}_i \neq y_i]
$$

where $w_{r_i}$ is the harm weight for risk tier $r_i$.

**Default weights:** high=10x, medium=3x, low=1x

In [ ]:
# Compute weighted harm loss for all three models
harm_balanced = weighted_harm_loss(y_true, y_pred_balanced, risk_tiers)
harm_high_errors = weighted_harm_loss(y_true, y_pred_high_risk_errors, risk_tiers)
harm_low_errors = weighted_harm_loss(y_true, y_pred_low_risk_errors, risk_tiers)

print("Weighted Harm Loss (Default Weights: high=10x, medium=3x, low=1x)")
print("="*70)
print(f"{'Model':<25} {'Accuracy':<15} {'Harm Loss':<15}")
print("-"*70)
print(f"{'Balanced errors':<25} {(y_pred_balanced == y_true).mean():<15.1%} {harm_balanced:<15.4f}")
print(f"{'High-risk errors':<25} {(y_pred_high_risk_errors == y_true).mean():<15.1%} {harm_high_errors:<15.4f}")
print(f"{'Low-risk errors':<25} {(y_pred_low_risk_errors == y_true).mean():<15.1%} {harm_low_errors:<15.4f}")

print("\nKey Insight:")
print(f"  High-risk errors model has {harm_high_errors / harm_low_errors:.1f}x higher harm")
print(f"  despite similar accuracy!")
print(f"  Accuracy alone is misleading for safety-critical systems")

### 3.1 Custom Harm Weights

Customize harm weights based on your clinical context.

In [ ]:
# Define custom harm weights (e.g., emergency department context)
custom_weights = {
    'high': 20.0,     # Missing critical case is 20x worse than low-risk error
    'medium': 5.0,    # Medium-risk errors 5x worse
    'low': 1.0,       # Baseline
}

# Recompute with custom weights
custom_harm_high_errors = weighted_harm_loss(
    y_true, y_pred_high_risk_errors, risk_tiers, harm_weights=custom_weights
)
custom_harm_low_errors = weighted_harm_loss(
    y_true, y_pred_low_risk_errors, risk_tiers, harm_weights=custom_weights
)

print("Custom Harm Weights (ED Context: high=20x, medium=5x, low=1x)")
print("="*70)
print(f"High-risk errors model: {custom_harm_high_errors:.4f}")
print(f"Low-risk errors model:  {custom_harm_low_errors:.4f}")
print(f"\nRatio: {custom_harm_high_errors / custom_harm_low_errors:.1f}x difference")
print(f"\n✓ Custom weights amplify the importance of high-risk accuracy")

## 4. Per-Tier Harm Analysis

Examine harm distribution across risk tiers to identify where errors concentrate.

In [ ]:
# Compute harm by tier for all models
tier_harm_balanced = harm_by_risk_tier(y_true, y_pred_balanced, risk_tiers)
tier_harm_high_errors = harm_by_risk_tier(y_true, y_pred_high_risk_errors, risk_tiers)
tier_harm_low_errors = harm_by_risk_tier(y_true, y_pred_low_risk_errors, risk_tiers)

print("Harm Distribution by Risk Tier")
print("="*80)
print(f"{'Model':<25} {'High':<15} {'Medium':<15} {'Low':<15}")
print("-"*80)

for model_name, tier_harm in [
    ('Balanced', tier_harm_balanced),
    ('High-risk errors', tier_harm_high_errors),
    ('Low-risk errors', tier_harm_low_errors)
]:
    high_h = tier_harm.get('high', 0)
    med_h = tier_harm.get('medium', 0)
    low_h = tier_harm.get('low', 0)
    print(f"{model_name:<25} {high_h:<15.4f} {med_h:<15.4f} {low_h:<15.4f}")

print("\nVisualize harm by tier:")

In [ ]:
# Visualize harm by tier for high-risk errors model
fig, ax = plot_harm_by_tier(
    tier_harm_high_errors,
    title="Harm Distribution: High-Risk Errors Model"
)
plt.show()

print("✓ Harm by tier visualized")
print("\nObservation: Most harm concentrated in high-risk tier (dangerous!)")

## 5. Escalation Failure Analysis

**Escalation failures** are the most critical safety concern:
- **Escalation failure**: Missing a high-risk case (false negative in high tier)
- **False escalation**: Over-escalating a low-risk case (false positive in low tier)

In [ ]:
# Analyze escalation failures for all models
esc_balanced = escalation_failure_analysis(y_true, y_pred_balanced, risk_tiers)
esc_high_errors = escalation_failure_analysis(y_true, y_pred_high_risk_errors, risk_tiers)
esc_low_errors = escalation_failure_analysis(y_true, y_pred_low_risk_errors, risk_tiers)

print("Escalation Failure Analysis")
print("="*80)
print(f"{'Model':<25} {'Escalation Failures':<25} {'False Escalations':<25}")
print("-"*80)

for model_name, esc in [
    ('Balanced', esc_balanced),
    ('High-risk errors', esc_high_errors),
    ('Low-risk errors', esc_low_errors)
]:
    print(f"{model_name:<25} {esc['escalation_failures']:<25} {esc['false_escalations']:<25}")

print("\nCritical Safety Metric:")
print(f"  High-risk errors model: {esc_high_errors['escalation_failures']} missed critical cases!")
print(f"  Low-risk errors model:  {esc_low_errors['escalation_failures']} missed critical cases")

In [ ]:
# Visualize escalation analysis for high-risk errors model
fig, axes = plot_escalation_analysis(
    esc_high_errors['escalation_failures'],
    esc_high_errors['false_escalations'],
    esc_high_errors['high_risk_samples'],
    esc_high_errors['low_risk_samples']
)
plt.show()

print("✓ Escalation analysis visualized")

## 6. Harm Concentration Index

**Harm concentration** measures what fraction of total harm occurs in the high-risk tier.

- **High concentration (>0.7)**: Errors predominantly in high-risk cases (CRITICAL SAFETY ISSUE)
- **Low concentration (<0.3)**: Errors distributed across tiers

In [ ]:
# Compute harm concentration for all models
conc_balanced = harm_concentration_index(y_true, y_pred_balanced, risk_tiers)
conc_high_errors = harm_concentration_index(y_true, y_pred_high_risk_errors, risk_tiers)
conc_low_errors = harm_concentration_index(y_true, y_pred_low_risk_errors, risk_tiers)

print("Harm Concentration Index")
print("="*70)
print(f"{'Model':<30} {'Concentration Index':<20} {'Interpretation':<20}")
print("-"*70)

for model_name, conc in [
    ('Balanced', conc_balanced),
    ('High-risk errors', conc_high_errors),
    ('Low-risk errors', conc_low_errors)
]:
    if conc > 0.7:
        interp = "CRITICAL"
    elif conc > 0.4:
        interp = "Moderate"
    else:
        interp = "Balanced"
    print(f"{model_name:<30} {conc:<20.2%} {interp:<20}")

print("\nRed Flag:")
print(f"  High-risk errors model has {conc_high_errors:.0%} harm in high-risk tier!")
print(f"  This indicates systematic failure on critical cases")

In [ ]:
# Visualize harm concentration for high-risk errors model
fig, ax = plot_harm_concentration(
    tier_harm_high_errors,
    conc_high_errors,
    figsize=(10, 7)
)
plt.show()

print("✓ Harm concentration visualized")

## 7. Comprehensive Harm Metrics

Use `compute_harm_metrics()` for a complete harm-aware analysis.

In [ ]:
# Compute comprehensive harm metrics
harm_metrics = compute_harm_metrics(
    y_true,
    y_pred_high_risk_errors,
    risk_tiers
)

print("Comprehensive Harm-Aware Metrics")
print("="*70)
print(f"Weighted Harm Loss:       {harm_metrics.weighted_harm_loss:.4f}")
print(f"Escalation Failures:      {harm_metrics.escalation_failures}")
print(f"False Escalations:        {harm_metrics.false_escalations}")
print(f"Harm Concentration:       {harm_metrics.harm_concentration:.2%}")

print("\nHarm by Tier:")
for tier, harm in harm_metrics.harm_by_tier.items():
    print(f"  {tier:>8}: {harm:.4f}")

print("\n✓ Comprehensive harm analysis complete")

## 8. Asymmetric Cost Matrices

For advanced cost modeling, use asymmetric cost matrices where different confusion matrix entries have different costs.

In [ ]:
# Define asymmetric costs
# False negative (missed intervention) is 10x worse than false positive
cost_balanced = asymmetric_cost_matrix(
    y_true, y_pred_balanced,
    cost_fn=10.0,  # False negative cost
    cost_fp=1.0,   # False positive cost
    cost_tn=0.0,   # True negative cost
    cost_tp=0.0    # True positive cost
)

cost_high_errors = asymmetric_cost_matrix(
    y_true, y_pred_high_risk_errors,
    cost_fn=10.0,
    cost_fp=1.0,
    cost_tn=0.0,
    cost_tp=0.0
)

cost_low_errors = asymmetric_cost_matrix(
    y_true, y_pred_low_risk_errors,
    cost_fn=10.0,
    cost_fp=1.0,
    cost_tn=0.0,
    cost_tp=0.0
)

print("Asymmetric Cost Analysis (FN cost = 10x FP cost)")
print("="*70)
print(f"{'Model':<30} {'Average Cost':<20}")
print("-"*70)
print(f"{'Balanced':<30} {cost_balanced:<20.4f}")
print(f"{'High-risk errors':<30} {cost_high_errors:<20.4f}")
print(f"{'Low-risk errors':<30} {cost_low_errors:<20.4f}")

print("\nInterpretation:")
print(f"  Lower cost is better")
print(f"  Cost framework can be customized to your clinical context")

## 9. Multi-Model Harm Comparison

Compare harm profiles across multiple CDSS models.

In [ ]:
# Create comparison table
comparison = pd.DataFrame({
    'Model': ['Balanced', 'High-Risk Errors', 'Low-Risk Errors'],
    'Accuracy': [
        (y_pred_balanced == y_true).mean(),
        (y_pred_high_risk_errors == y_true).mean(),
        (y_pred_low_risk_errors == y_true).mean()
    ],
    'Harm Loss': [harm_balanced, harm_high_errors, harm_low_errors],
    'Escalation Failures': [
        esc_balanced['escalation_failures'],
        esc_high_errors['escalation_failures'],
        esc_low_errors['escalation_failures']
    ],
    'Harm Concentration': [conc_balanced, conc_high_errors, conc_low_errors]
})

# Sort by harm loss (lower is better)
comparison = comparison.sort_values('Harm Loss')

print("Multi-Model Harm-Aware Comparison")
print("="*90)
print(comparison.to_string(index=False))

print("\n✓ Best model: Lowest harm loss with acceptable escalation failure rate")
print("✓ Worst model: High harm concentration and escalation failures")

## 10. Summary and Key Takeaways

### Core Concepts:

1. **Weighted Harm Loss**: Weight errors by risk tier (high >> medium > low)
2. **Per-Tier Analysis**: Identify where errors concentrate
3. **Escalation Failures**: Most critical safety metric (missed high-risk cases)
4. **Harm Concentration**: Fraction of harm in high-risk tier
5. **Asymmetric Costs**: Custom cost matrices for context-specific evaluation

### Key API Functions:

```python
# Weighted harm
harm_loss = weighted_harm_loss(y_true, y_pred, risk_tiers, harm_weights)
tier_harm = harm_by_risk_tier(y_true, y_pred, risk_tiers)

# Escalation analysis
esc_analysis = escalation_failure_analysis(y_true, y_pred, risk_tiers)

# Harm concentration
concentration = harm_concentration_index(y_true, y_pred, risk_tiers)

# Comprehensive metrics
harm_metrics = compute_harm_metrics(y_true, y_pred, risk_tiers)
```

### Next Steps:

- **[Notebook 05](05_end_to_end_pipeline.ipynb)**: Complete evaluation workflow integrating all metrics

### Clinical Impact:

Harm-aware evaluation ensures:
- **Prioritize high-risk accuracy**: Errors where they matter most
- **Safety-first deployment**: Identify unsafe error patterns
- **Contextual evaluation**: Custom harm weights for your clinical setting